In [1]:
!pip install wittgenstein pandas scikit-learn

import pandas as pd
from sklearn.datasets import load_iris
import wittgenstein as lw

# ---- Step 1: Load dataset (Iris for demo) ----

iris = load_iris(as_frame=True)

X = iris.data
y = iris.target

df = pd.concat([X, y.rename("target")], axis=1)

print("Dataset head:\n", df.head())


# ---- Step 2: RIPPER Algorithm (via wittgenstein) ----

model = lw.RIPPER()

# Specify the positive class, for example, class 0 (setosa)
model.fit(X, y, pos_class=0)

print("\n=== RIPPER Rules ===")
print(model.ruleset_)


# ---- Step 3: Simplified FOIL Implementation ----
# FOIL tries to build rules like "IF ... THEN class"
# We'll demonstrate with binary classification subset (setosa vs not-setosa)

df_bin = df.copy()

df_bin['target'] = (df_bin['target'] == 0).astype(int)
# 1 = setosa, 0 = others


# Candidate literals
attributes = list(X.columns)


def foil_gain(pos_before, neg_before, pos_after, neg_after):
    # FOIL information gain formula

    if pos_after == 0:
        return -1e9

    return pos_after * (
        (pos_after / (pos_after + neg_after)) -
        (pos_before / (pos_before + neg_before))
    )


def foil(df, target_col='target'):

    rules = []

    pos_total = df[target_col].sum()
    neg_total = len(df) - pos_total

    while pos_total > 0:

        rule = []

        pos_rem = pos_total
        neg_rem = neg_total

        covered = df.copy()

        while neg_rem > 0:

            best_gain, best_attr = -1e9, None

            for attr in attributes:

                for val in df[attr].unique():

                    subset = covered[covered[attr] == val]

                    pos_after = subset[target_col].sum()
                    neg_after = len(subset) - pos_after

                    gain = foil_gain(
                        pos_rem,
                        neg_rem,
                        pos_after,
                        neg_after
                    )

                    if gain > best_gain:
                        best_gain, best_attr, best_val, \
                        best_subset = gain, attr, val, subset

            if best_attr is None:
                break

            rule.append((best_attr, best_val))

            covered = best_subset

            pos_rem = covered[target_col].sum()
            neg_rem = len(covered) - pos_rem

        rules.append((rule, 1))

        # Remove covered examples
        df = df.drop(covered.index)

        pos_total = df[target_col].sum()
        neg_total = len(df) - pos_total

    return rules


rules = foil(df_bin)

print("\n=== FOIL Learned Rules (Setosa vs Not) ===")

for conds, pred in rules:

    cond_str = " AND ".join(
        [f"{a}={v}" for a, v in conds]
    )

    print(f"IF {cond_str} THEN class={pred}")

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/78.3 kB ? eta -:--:--
     ---------- ----------------------------- 20.5/78.3 kB ? eta -:--:--
     -------------- ----------------------- 30.7/78.3 kB 435.7 kB/s eta 0:00:01
     ------------------------ ------------- 51.2/78.3 kB 327.7 kB/s eta 0:00:01
     ---------------------------------- --- 71.7/78.3 kB 357.2 kB/s eta 0:00:01
     -------------------------------------- 78.3/78.3 kB 333.5 kB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for wittgenstein: filename=wittgenstein-0.3.5-py3-none-any.whl size=99038 sha256=93962fcbc4eb538eb32ac3a06b9c03a970db1aa502ed27ffeac7fe204b97f581
  Stored in directory: c:\users\sushm\appdata\local\pip\cache\wheels\b9\8f\6c\da8540b89849f36a9b2500f30a9aa42d6464fa694cbe78d041
Successfully built wittgenstein


C:\Users\sushm\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Dataset head:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

=== RIPPER Rules ===
[[petalwidth(cm)=<0.2] V [petalwidth(cm)=0.2-0.4] V [petallength(cm)=1.5-1.7]]

=== FOIL Learned Rules (Setosa vs Not) ===
IF petal width (cm)=0.2 THEN class=1
IF petal width (cm)=0.4 THEN class=1
IF petal width (cm)=0.3 THEN class=1
IF petal width (cm)=0.1 THEN class=1
IF sepal width (cm)=3.5 THEN class=1
IF petal length (cm)=1.7 THEN class=1
